# Qixia ROCm LoRA Train

直接使用 LLaMA Factory 在 ROCm AMD GPU 上做一次 LoRA 微调。这个 notebook 不需要额外的大模型 API key。

默认数据（聊天格式 ShareGPT，由阶跃 API 从原著台词改造而来）：

- `data/qixia_chat_train.json`: 6248 条聊天训练数据（用户问 → 齐夏答）
- `data/qixia_chat_valid.json`: 329 条验证数据
- `data/dataset_info.json`: LLaMA Factory 数据集配置（dataset 名仍为 `qixia_train` / `qixia_valid`）


## 0. 参数区

在这里修改：模型、`TRAIN_OVERRIDES`、`RUN_TAG`。改完后重跑 cell 2，再跑 runtime config + train cell 即可生效，不需要改 yaml 文件。

想 A/B 一次新的训练参数，把 `RUN_TAG` 改成 `b3` / `r16` 之类，`output_dir` 会带上该后缀，避免覆盖旧 LoRA。


In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

def find_repo_dir(start: Path) -> Path:
    cloud_repo = Path('/workspace/repo')
    candidates = [cloud_repo, start, *start.parents]
    for path in candidates:
        if (path / 'configs').is_dir() and (path / 'data').is_dir() and (path / 'scripts').is_dir():
            return path
    raise RuntimeError(f'Could not find repo root from {start}')

REPO_DIR = find_repo_dir(Path.cwd().resolve())
PERSIST_DIR = Path('/network-workspace')

MODEL_IDS = {
    'qwen3_5_9b': 'Qwen/Qwen3.5-9B',
    'qwen3_6_35b_a3b': 'Qwen/Qwen3.6-35B-A3B',
}
MODEL_CHOICE = 'qwen3_5_9b'  # 可改为 'qwen3_6_35b_a3b'
MODEL_ID = MODEL_IDS[MODEL_CHOICE]

# 训练参数。改完后重跑 "runtime config" 那个 cell 即可生效，不需要改 yaml 文件。
TRAIN_OVERRIDES = {
    'per_device_train_batch_size': 4,
    'gradient_accumulation_steps': 4,
    'learning_rate': 1.0e-4,
    'num_train_epochs': 1.0,
    'lora_rank': 8,
    'lora_alpha': 16,
    'lora_dropout': 0.05,
    'cutoff_len': 1024,
    'warmup_ratio': 0.05,
    'lr_scheduler_type': 'cosine',
    'logging_steps': 5,
    'save_steps': 100,
    'eval_steps': 100,
    'gradient_checkpointing': False,
}

# 想 A/B 不同参数时把 RUN_TAG 改成 'b3'/'r16' 之类，output_dir 会带上这个后缀，避免覆盖旧 LoRA。
RUN_TAG = ''

RUN_NAME = f'{MODEL_CHOICE}_lora' + (f'_{RUN_TAG}' if RUN_TAG else '')
CONFIG_PATH = REPO_DIR / 'configs' / f'{MODEL_CHOICE}_lora.yaml'
DATA_DIR = REPO_DIR / 'data'
OUTPUT_DIR = PERSIST_DIR / 'outputs' / RUN_NAME
LOG_DIR = PERSIST_DIR / 'logs'
MODEL_CACHE_DIR = PERSIST_DIR / 'models'
RUNTIME_CONFIG_DIR = PERSIST_DIR / 'runtime_configs'
RUNTIME_CONFIG_PATH = RUNTIME_CONFIG_DIR / f'{RUN_NAME}.yaml'

RUN_INSTALL = True
RUN_TRAIN = False

assert CONFIG_PATH.exists(), f'missing config: {CONFIG_PATH}'
assert DATA_DIR.exists(), f'missing data dir: {DATA_DIR}'

for path in [PERSIST_DIR, OUTPUT_DIR, LOG_DIR, MODEL_CACHE_DIR, RUNTIME_CONFIG_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print('python:', sys.version)
print('repo:', REPO_DIR)
print('persist:', PERSIST_DIR)
print('model_choice:', MODEL_CHOICE)
print('model_id:', MODEL_ID)
print('run_name:', RUN_NAME)
print('config:', CONFIG_PATH)
print('runtime_config:', RUNTIME_CONFIG_PATH)
print('data:', DATA_DIR)
print('model_cache:', MODEL_CACHE_DIR)
print('output:', OUTPUT_DIR)
print('overrides:')
for k, v in TRAIN_OVERRIDES.items():
    print(f'  {k}: {v}')
eff = TRAIN_OVERRIDES['per_device_train_batch_size'] * TRAIN_OVERRIDES['gradient_accumulation_steps']
print(f'  effective_batch_size: {eff}')


## 1. 命令工具


In [ ]:
def run(cmd, cwd=None, timeout=None):
    if cwd is None:
        cwd = REPO_DIR

    print(f'$ {cmd}', flush=True)
    process = subprocess.Popen(
        cmd,
        shell=True,
        cwd=cwd,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        bufsize=1,
    )
    try:
        for line in process.stdout:
            print(line, end='', flush=True)
        returncode = process.wait(timeout=timeout)
    except Exception:
        process.kill()
        raise
    if returncode != 0:
        raise RuntimeError(f'command failed with exit code {returncode}: {cmd}')
    return returncode


## 2. ROCm / PyTorch 自检

必须看到：

- `torch.version.hip` 有版本号
- `torch.cuda.is_available()` 是 `True`
- GPU 显存符合预期


In [ ]:
run('rocm-smi --showproductname --showmeminfo vram --showuse || true', timeout=60)

import torch
print('torch:', torch.__version__)
print('hip:', getattr(torch.version, 'hip', None))
print('cuda available:', torch.cuda.is_available())
print('device count:', torch.cuda.device_count())
assert getattr(torch.version, 'hip', None), '当前不是 ROCm/HIP 版 PyTorch，先不要继续。'
assert torch.cuda.is_available(), 'PyTorch 没有识别到 AMD GPU。'

props = torch.cuda.get_device_properties(0)
print('device:', torch.cuda.get_device_name(0))
print('total_memory_gb:', round(props.total_memory / 1024**3, 2))
print('bf16 supported:', torch.cuda.is_bf16_supported())


## 3. 安装训练依赖

`requirements-rocm.txt` 不包含 `torch`，避免覆盖云镜像已有的 ROCm PyTorch。


In [ ]:
REQ_PATH = REPO_DIR / 'requirements-rocm.txt'
assert REQ_PATH.exists(), f'missing requirements file: {REQ_PATH}'

if RUN_INSTALL:
    run('python -m pip install -U pip setuptools wheel', timeout=300)
    run(f'python -m pip install -r {REQ_PATH} --upgrade-strategy only-if-needed', timeout=1200)
else:
    print('skip install')

run('python scripts/patch_llamafactory_qwen35_text.py', timeout=60)

import torch
print('torch after install:', torch.__version__)
print('hip after install:', getattr(torch.version, 'hip', None))
assert getattr(torch.version, 'hip', None), '安装依赖后 torch 不再是 ROCm 版。'


## 4. 数据校验

这里读的是仓库内完整数据，不是 sample。


In [ ]:
run('python scripts/validate_dataset.py data/qixia_chat_train.json data/qixia_chat_valid.json')

train = json.loads((DATA_DIR / 'qixia_chat_train.json').read_text(encoding='utf-8'))
valid = json.loads((DATA_DIR / 'qixia_chat_valid.json').read_text(encoding='utf-8'))
print('train examples:', len(train))
print('valid examples:', len(valid))
print(json.dumps(train[0], ensure_ascii=False, indent=2)[:1200])


## 5. 训练配置检查


In [ ]:
def replace_yaml_value(text: str, key: str, value) -> str:
    if isinstance(value, float):
        formatted = repr(value)
    else:
        formatted = str(value)
    lines = []
    replaced = False
    for line in text.splitlines():
        if line.startswith(f'{key}:'):
            lines.append(f'{key}: {formatted}')
            replaced = True
        else:
            lines.append(line)
    if not replaced:
        # 没有该 key 时追加到末尾，便于将来新增参数。
        lines.append(f'{key}: {formatted}')
    return '\n'.join(lines) + '\n'

assert CONFIG_PATH.exists()
assert (DATA_DIR / 'dataset_info.json').exists()
assert (DATA_DIR / 'qixia_chat_train.json').exists()
assert (DATA_DIR / 'qixia_chat_valid.json').exists()

print('downloading/checking local model from ModelScope:', MODEL_ID)
from modelscope import snapshot_download
MODEL_DIR = Path(snapshot_download(MODEL_ID, cache_dir=str(MODEL_CACHE_DIR))).resolve()
assert (MODEL_DIR / 'config.json').exists(), f'missing model config.json: {MODEL_DIR}'

runtime_config = CONFIG_PATH.read_text(encoding='utf-8')
runtime_config = replace_yaml_value(runtime_config, 'model_name_or_path', MODEL_DIR)
runtime_config = replace_yaml_value(runtime_config, 'output_dir', OUTPUT_DIR)
runtime_config = replace_yaml_value(runtime_config, 'logging_dir', LOG_DIR)

# 用 notebook 顶部 TRAIN_OVERRIDES 覆盖 yaml 默认值。
for key, value in TRAIN_OVERRIDES.items():
    runtime_config = replace_yaml_value(runtime_config, key, value)

RUNTIME_CONFIG_PATH.write_text(runtime_config, encoding='utf-8')

print('model_dir:', MODEL_DIR)
print('runtime config:')
print(RUNTIME_CONFIG_PATH.read_text(encoding='utf-8'))

import yaml
runtime_params = yaml.safe_load(RUNTIME_CONFIG_PATH.read_text(encoding='utf-8'))
train_bs = int(runtime_params.get('per_device_train_batch_size', 1))
grad_accum = int(runtime_params.get('gradient_accumulation_steps', 1))
print('effective_batch_size:', train_bs * grad_accum)
print('progress_bar:', 'enabled' if runtime_params.get('disable_tqdm') is False else 'default')


## 6. 开始 LoRA 训练

确认前面都通过后，把参数区的 `RUN_TRAIN = True` 再运行这一格。


In [ ]:
if RUN_TRAIN:
    env = os.environ.copy()
    env['HIP_VISIBLE_DEVICES'] = '0'
    env['CUDA_VISIBLE_DEVICES'] = '0'
    env['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'
    env['TOKENIZERS_PARALLELISM'] = 'false'
    env['PYTHONUNBUFFERED'] = '1'
    run(f'python scripts/train_lora.py --config {RUNTIME_CONFIG_PATH}', timeout=None)
else:
    print('skip train. Set RUN_TRAIN=True after checking ROCm, deps, data, and config.')


## 7. 快速推理

训练完成后，把 `--adapter` 指向 LoRA 输出目录。


In [ ]:
adapter = OUTPUT_DIR
base_model = globals().get('MODEL_DIR', MODEL_ID)
print('adapter:', adapter)
print('base_model:', base_model)
print('run after training:')
print(f'python scripts/quick_infer.py --model {base_model} --adapter {adapter} --prompt "如果一个规则看起来互相矛盾，你会怎么判断？"')


## 8. 打包 LoRA 到可下载目录

把最新 LoRA checkpoint 打包到 `/workspace/repo/downloads`，JupyterLab 左侧文件浏览器可以直接下载。


In [ ]:
import re
import shutil
import tarfile


def checkpoint_step(path: Path) -> int:
    match = re.search(r'checkpoint-(\d+)$', path.name)
    return int(match.group(1)) if match else -1


checkpoints = [p for p in OUTPUT_DIR.glob('checkpoint-*') if (p / 'adapter_model.safetensors').exists()]
ADAPTER_DIR = max(checkpoints, key=checkpoint_step) if checkpoints else OUTPUT_DIR
assert (ADAPTER_DIR / 'adapter_model.safetensors').exists(), f'missing adapter in {ADAPTER_DIR}'
assert (ADAPTER_DIR / 'adapter_config.json').exists(), f'missing adapter config in {ADAPTER_DIR}'

DOWNLOAD_DIR = REPO_DIR / 'downloads'
PACKAGE_DIR = DOWNLOAD_DIR / f'{RUN_NAME}_{ADAPTER_DIR.name}_infer'
TAR_PATH = DOWNLOAD_DIR / f'{RUN_NAME}_{ADAPTER_DIR.name}_infer.tar.gz'

if PACKAGE_DIR.exists():
    shutil.rmtree(PACKAGE_DIR)
PACKAGE_DIR.mkdir(parents=True, exist_ok=True)

files = [
    'adapter_config.json',
    'adapter_model.safetensors',
    'chat_template.jinja',
    'processor_config.json',
    'tokenizer.json',
    'tokenizer_config.json',
    'README.md',
]
for name in files:
    src = ADAPTER_DIR / name
    if src.exists():
        shutil.copy2(src, PACKAGE_DIR / name)

with tarfile.open(TAR_PATH, 'w:gz') as tar:
    tar.add(PACKAGE_DIR, arcname=PACKAGE_DIR.name)

print('adapter_dir:', ADAPTER_DIR)
print('package_dir:', PACKAGE_DIR)
print('download_file:', TAR_PATH)
print('size_mb:', round(TAR_PATH.stat().st_size / 1024 / 1024, 2))
print('在 JupyterLab 左侧刷新并下载:', TAR_PATH.relative_to(REPO_DIR))


## 9. Gradio 对话测试

加载 base model + 最新 LoRA checkpoint，启动 Gradio + Cloudflare 隧道。

- 第一次运行会自动下载 `cloudflared` 二进制到 `/opt/tunnels`。
- 隧道启动后会打印一个 `https://...trycloudflare.com` 地址，**从你本地浏览器打开就能聊天**。
- 这个临时公网地址 72 小时有效，重跑 cell 会刷新。
- 不需要手动开第二个终端，也不需要云平台端口转发功能。


In [ ]:
import re
import torch
from peft import PeftModel
from transformers import AutoTokenizer, Qwen3_5ForCausalLM

try:
    import gradio as gr
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'gradio'])
    import gradio as gr


def checkpoint_step(path: Path) -> int:
    match = re.search(r'checkpoint-(\d+)$', path.name)
    return int(match.group(1)) if match else -1


checkpoints = [p for p in OUTPUT_DIR.glob('checkpoint-*') if (p / 'adapter_model.safetensors').exists()]
ADAPTER_DIR = max(checkpoints, key=checkpoint_step) if checkpoints else OUTPUT_DIR

BASE_MODEL = Path(globals().get('MODEL_DIR', MODEL_CACHE_DIR / 'Qwen' / 'Qwen3___5-9B'))
if not BASE_MODEL.exists():
    BASE_MODEL = MODEL_ID

print('base_model:', BASE_MODEL)
print('adapter:', ADAPTER_DIR)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
base = Qwen3_5ForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.bfloat16,
    device_map='auto',
    trust_remote_code=True,
)
model = PeftModel.from_pretrained(base, ADAPTER_DIR)
model.eval()


def clean_reply(text: str) -> str:
    # 模型经常续写 "user", "user:", "用户:", "<|im_xxx|>"，全部截掉。
    import re as _re
    cut = len(text)
    patterns = [
        r'(?i)\buser\s*[:：]',     # user: / User：
        r'(?i)\bassistant\s*[:：]',
        r'用户\s*[:：]',
        r'齐夏\s*[:：]',
        r'\n\s*(?:user|User|用户|assistant|Assistant|齐夏)\b',
        r'<\|im_(?:start|end)\|>',
    ]
    for pat in patterns:
        m = _re.search(pat, text)
        if m and m.start() < cut:
            cut = m.start()
    # 还要单独处理裸的 \buser\b / \bassistant\b（防止模型只吐 "user" 不带冒号）
    bare = _re.search(r'(?i)(?<![A-Za-z])(?:user|assistant)(?![A-Za-z])', text)
    if bare and bare.start() < cut:
        cut = bare.start()
    return text[:cut].rstrip(' \n\t：:，,。.').strip()


def qixia_reply(question: str, max_new_tokens: int = 120, temperature: float = 0.7) -> str:
    prompt = (
        '你正在扮演《十日终焉》中的齐夏。保持冷静、克制、善于观察和推理。\n'
        '只回答下面这个问题，不要续写 user/assistant，不要展开新对话。\n\n'
        f'用户：{question}\n齐夏：'
    )
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    generation_kwargs = {
        'max_new_tokens': max_new_tokens,
        'do_sample': temperature > 0,
        'top_p': 0.9,
        'repetition_penalty': 1.15,
        'eos_token_id': tokenizer.eos_token_id,
        'pad_token_id': tokenizer.eos_token_id,
    }
    if temperature > 0:
        generation_kwargs['temperature'] = temperature
    with torch.no_grad():
        output = model.generate(**inputs, **generation_kwargs)
    text = tokenizer.decode(output[0][inputs.input_ids.shape[-1]:], skip_special_tokens=True)
    return clean_reply(text)


def chat_fn(message, history, max_new_tokens, temperature):
    return qixia_reply(message, max_new_tokens=int(max_new_tokens), temperature=float(temperature))


EXAMPLES = [
    ['如果一个规则看起来互相矛盾，你会怎么判断？', 120, 0.7],
    ['我现在很慌，怎么办？', 120, 0.7],
    ['你最讨厌什么样的人？', 120, 0.7],
]

demo = gr.ChatInterface(
    fn=chat_fn,
    title='齐夏 LoRA 对话测试',
    description=f'base: {BASE_MODEL}  |  adapter: {ADAPTER_DIR}',
    additional_inputs=[
        gr.Slider(32, 512, value=120, step=8, label='max_new_tokens'),
        gr.Slider(0.0, 1.5, value=0.7, step=0.05, label='temperature'),
    ],
    examples=EXAMPLES,
)



# 云端没有端口转发时，用 Cloudflare 隧道出公网地址。72 小时有效。
import subprocess
import time
import signal
import os

CLOUDFLARED_PATH = Path('/opt/tunnels/cloudflared')
GRADIO_PORT = 7860
TUNNEL_LOG = Path('/tmp/cloudflared_tunnel.log')

if not CLOUDFLARED_PATH.exists() or CLOUDFLARED_PATH.stat().st_size < 30_000_000:
    CLOUDFLARED_PATH.parent.mkdir(parents=True, exist_ok=True)
    print('downloading cloudflared ...')
    subprocess.check_call([
        'curl', '-L', '--max-time', '180', '-o', str(CLOUDFLARED_PATH),
        'https://gh-proxy.org/https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',
    ])
    CLOUDFLARED_PATH.chmod(0o755)
    print('cloudflared ready:', CLOUDFLARED_PATH)

# Kill any existing tunnel on this port
subprocess.run(['pkill', '-f', f'cloudflared.*:{GRADIO_PORT}'], capture_output=True)
time.sleep(0.5)

# Start tunnel in background
proc = subprocess.Popen(
    [str(CLOUDFLARED_PATH), 'tunnel', '--url', f'http://localhost:{GRADIO_PORT}'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    preexec_fn=os.setsid,
)

public_url = None
deadline = time.time() + 20
print('waiting for tunnel URL ...')
while time.time() < deadline:
    line = proc.stdout.readline()
    if not line:
        continue
    print(line.rstrip())
    if 'Your quick Tunnel has been created' in line:
        # next line contains URL
        while time.time() < deadline:
            next_line = proc.stdout.readline()
            if not next_line:
                continue
            print(next_line.rstrip())
            if next_line.startswith('202') and '.trycloudflare.com' in next_line:
                import re
                match = re.search(r'https://[\w\-]+\.trycloudflare\.com', next_line)
                if match:
                    public_url = match.group(0)
                    break
        break

if public_url:
    print('\n' + '='*80)
    print('✅ 在你本地浏览器打开：')
    print(public_url)
    print('='*80 + '\n')
else:
    print('⚠️  未解析到隧道 URL，手动查看 cloudflared 输出。')

# Keep this proc object alive, cell exits later when Gradio is killed.
TUNNEL_PROC = proc


demo.queue().launch(
    server_name='0.0.0.0',
    server_port=GRADIO_PORT,
    share=False,
    inline=False,
)


## 10. 可选合并 LoRA

如果需要导出合并后的完整模型：


In [ ]:
merged_out = PERSIST_DIR / 'outputs' / f'{MODEL_CHOICE}_merged'
base_model = globals().get('MODEL_DIR', MODEL_ID)
print(f'python scripts/merge_lora.py --model {base_model} --adapter {OUTPUT_DIR} --out {merged_out}')
